In [ ]:
# Load packages
import os
import numpy as np
import pandas as pd 
import rpy2.robjects as ro
from rpy2.robjects.vectors import IntVector, FloatVector, StrVector
from rpy2.robjects.packages import importr
from scipy.sparse import coo_matrix

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
    "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

In [ ]:
# Load SAGIE-GENE sparse GRM 
bucket=f'{os.environ["WORKSPACE_BUCKET"]}/data/saige_step00/grm/EUR_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx'
grm = pd.read_csv(bucket, sep=' ') 

# Load SAIGE-GENE sparse GRM sample ids 
bucket=f'{os.environ["WORKSPACE_BUCKET"]}/data/saige_step00/grm/EUR_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt'
sampleids = pd.read_csv(bucket, sep='\t', names=['id']) 

# Add sample index 
sampleids['index'] = range(1,len(sampleids) + 1)

# Reformat sparse GRM columns 
grm.columns = ['id1','id2','ibd','real','symmetric']
# Reset column index 
grm = grm.iloc[1:].reset_index()

print('======= CHECK SPARSE GRM =======')
print(grm[['id1','id2','ibd']].head())
print('======= CHECK SPARSE GRM =======')
print(grm[['id1','id2','ibd']].tail())

In [ ]:
# Create key of sample ids 
key = dict(zip(sampleids["index"], sampleids["id"]))

# Map sample ids onto index 
grm["id1"] = grm["id1"].astype('int')
grm["id1"] = grm["id1"].map(key)
# Map sample ids onto index 
grm["id2"] = grm["id2"].astype('int')
grm["id2"] = grm["id2"].map(key)

# Tranform sample ids to BGEN sample ids 
grm['id1'] = grm['id1'].astype(str) + '_' + grm['id1'].astype(str)
grm['id2'] = grm['id2'].astype(str) + '_' + grm['id2'].astype(str)

print('======= CHECK SPARSE GRM =======')
print(grm[['id1','id2','ibd']].head())


In [ ]:
# Load R Matrix package
Matrix = importr("Matrix")

# Get unique sample IDs
ids = np.unique(np.concatenate([
    grm["id1"].values,
    grm["id2"].values
]))

# Map sample IDs to matrix indices
id_to_idx = {x: i for i, x in enumerate(ids)}

rows = grm["id1"].map(id_to_idx).to_numpy()
cols = grm["id2"].map(id_to_idx).to_numpy()
values = grm["ibd"].to_numpy()

# Construct sparse matrix
G = coo_matrix(
    (values, (rows, cols)),
    shape=(len(ids), len(ids))
).tocsr()

# Ensure the matrix is symmetric
G = G.maximum(G.T)

print("Matrix dimensions:", G.shape)
print("Non-zero entries:", G.nnz)

# Convert to CSC for R's dgCMatrix representation
G = G.tocsc()

# R uses 1-based indices
r_i = IntVector(G.indices + 1)
r_p = IntVector(G.indptr)
r_x = FloatVector(G.data)

r_sparseGRM = Matrix.sparseMatrix(
    i=r_i,
    p=r_p,
    x=r_x,
    dims=ro.IntVector(G.shape)
)

# Put matrix and sample IDs into R
ro.globalenv["sparseGRM"] = r_sparseGRM
ro.globalenv["sample_names"] = StrVector([str(x) for x in ids])

# Assign row/column names and save
ro.r("""
    library(Matrix)

    rownames(sparseGRM) <- sample_names
    colnames(sparseGRM) <- sample_names

    print(head(sparseGRM))
    print(head(rownames(sparseGRM)))
    print(head(colnames(sparseGRM)))

    save(
        sparseGRM,
        file = "sparseGRM.RData"
    )
""")

In [ ]:
# Copy sparse GRM to workspace bucket 
!gcloud storage cp sparseGRM.RData {os.environ["WORKSPACE_BUCKET"]}/data/staar/grm/

In [ ]:
#XTRA CODE [CHECK SAMPLES]
from rpy2.robjects import r

value = r["["](r_sparseGRM, 218625, 125476)
print(value)

from rpy2.robjects import r

value = r["["](r_sparseGRM, 125476, 218625)
print(value)

In [ ]:
# Create key of sample ids 
key = dict(zip(sampleids["index"], sampleids["id"]))

# Map sample ids onto index 
grm["id1"] = grm["id1"].astype('int')
grm["id1"] = grm["id1"].map(key)
# Map sample ids onto index 
grm["id2"] = grm["id2"].astype('int')
grm["id2"] = grm["id2"].map(key)

# Create lsit of genetic relationships
upper = grm.copy()
upper = upper[(upper['id1'] != upper['id2'])]

# Create list of inverse genetic relationships
lower = grm.rename(
    columns={"id1": "id2", "id2": "id1"}
)

# Create symmetrical sparse GMR 
grm = pd.concat([upper, lower], ignore_index=True)

# Tranform sample ids to BGEN sample ids 
grm['id1'] = grm['id1'].astype(str) + '_' + grm['id1'].astype(str)
grm['id2'] = grm['id2'].astype(str) + '_' + grm['id2'].astype(str)

print('====== CHECK SPARSE GRM ======')
print(grm[['id1','id2','ibd']].head())

In [ ]:
# Get unique sample ids
ids = np.unique(np.concatenate([
    grm["id1"].values,
    grm["id2"].values
]))

# Map sample ids to matrix indices
id_to_idx = {x: i for i, x in enumerate(ids)}

rows = grm["id1"].map(id_to_idx).to_numpy()
cols = grm["id2"].map(id_to_idx).to_numpy()
values = grm["ibd"].to_numpy()

# Construct sparse matrix
G = coo_matrix(
    (values, (rows, cols)),
    shape=(len(ids), len(ids))
).tocsr()

# IDs corresponding to rows/columns
row_ids = ids
col_ids = ids

In [ ]:
# Convert of spase GRM  
sparseGRM = pd.DataFrame.sparse.from_spmatrix(
    G,
    index=ids,
    columns=ids
)

# Create list columns and rows for sparse GRM 
column_names = sparseGRM.columns
row_names = sparseGRM.index

Matrix = importr("Matrix")

G = G.tocsc()

# R uses 1-based indices
r_i = IntVector(G.indices + 1)
r_p = IntVector(G.indptr)
r_x = FloatVector(G.data)

r_sparseGRM = Matrix.sparseMatrix(
    i=r_i,
    p=r_p,
    x=r_x,
    dims=ro.IntVector(G.shape)
)

In [ ]:
# Put it into R
ro.globalenv["sparseGRM"] = r_sparseGRM
# Explicitly convert Python names -> R character vectors
ro.globalenv["column_names"] = StrVector([str(x) for x in column_names])
ro.globalenv["row_names"] = StrVector([str(x) for x in row_names])

# Save as RData
ro.r("library(Matrix)")
ro.r("sparseGRM <- as(sparseGRM, 'symmetricMatrix')")
ro.r("rownames(sparseGRM) <- row_names")
ro.r("colnames(sparseGRM) <- column_names")
ro.r("print(head(sparseGRM))")
ro.r("save(sparseGRM, file='sparseGRM.RData')")

In [ ]:
# Copy sparse GRM to workspace bucket 
!gcloud storage cp sparseGRM.RData {os.environ["WORKSPACE_BUCKET"]}/data/staar/grm/